# Figures

This notebook regenerates the figures of the paper and the figures for the other datasets that the paper refers to
(also stored in `figures/`), and shows them inline.

**Before running it:**
- put the datasets in `data/` (see `README.md`);
- fit the activity model on each dataset you want to plot: `python experiments/02_fit_activity_model.py --dataset <key>`;
- for Figure 1, also choose the EASE regularization: `python experiments/05_ease_lambda.py --dataset ml1m_r4`.

`run_all.sh` does all of this, together with every other result of the paper. Running the notebook needs Jupyter,
which is not in `requirements.txt` (`pip install notebook`).

In [ ]:
import os
import runpy
import shutil
import sys
from pathlib import Path

from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # the repository root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
FIGURES = ROOT / "results" / "figures"


def run_script(name, *args):
    """Run experiments/<name> as from the command line, with the given arguments."""
    argv = sys.argv
    sys.argv = [name, *args]
    try:
        runpy.run_path(str(ROOT / "experiments" / name), run_name="__main__")
    finally:
        sys.argv = argv

## Figure 1: MovieLens-1M

Lift, cosine, Pearson correlation and EASE weights for every pair of films with at least 50 positive ratings (in
blocks of 14 films, sorted by popularity) and the four tail buckets; rows: observed data, activity model,
independence model.

In [ ]:
run_script("06_main_figure.py", "--dataset", "ml1m_r4")
display(Image(filename=str(FIGURES / "ml1m_r4" / "paper_matrices_wide.png")))

## Other datasets (Appendix C)

All items in popularity groups that hold equal shares of interactions: block lift, block cosine and block Pearson
correlation for the observed data, the activity model and the independence model; for each dataset also the mean
popularity of a user's items as a function of the number of items they consumed. Amazon Books takes a few minutes.

In [ ]:
DATASETS = ["ml20m_r4", "lastfm2k", "amazon_books"]
for d in DATASETS:
    run_script("04_catalog_groups.py", "--datasets", d)
    for name in ["catalog_block_matrices", "catalog_profile_and_curves"]:
        print(d, name)
        display(Image(filename=str(FIGURES / d / f"{name}.png")))

## Copy into `figures/`

`figures/` holds the figures the paper refers to: the block matrices of every dataset, and the popularity profile
of Amazon Books.

In [ ]:
for d in DATASETS:
    (ROOT / "figures" / d).mkdir(parents=True, exist_ok=True)
    shutil.copy(FIGURES / d / "catalog_block_matrices.pdf", ROOT / "figures" / d / "catalog_block_matrices.pdf")
if "amazon_books" in DATASETS:
    shutil.copy(FIGURES / "amazon_books" / "catalog_profile_and_curves.pdf",
                ROOT / "figures" / "amazon_books" / "catalog_profile_and_curves.pdf")